# P48 — LoRA: adaptación de rango bajo de modelos de lenguaje grandes

## 1. Título y paper

**Paper:** *LoRA: Low-Rank Adaptation of Large Language Models*  
**Autoría:** Edward J. Hu, Yelong Shen, Phillip Wallis, y otros  
**Año y venue:** 2021 · arXiv:2106.09685 · ICLR 2022  
**Nivel:** L3 · **Motor:** `lora`  
**Ficha completa:** [`P48_lora`](../../papers/foundational/P48_lora/README.md)

**Hito:** Ajustar un modelo enorme entrenando una fracción diminuta de parámetros, sin coste añadido en inferencia.

- [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El ajuste fino completo exige una copia entera del modelo por tarea: inviable en almacenamiento y en memoria de entrenamiento cuando el modelo tiene miles de millones de parámetros.
2. Ejecutar una implementación mínima de la propuesta: Congelar los pesos originales y aprender una actualización factorizada de rango bajo, W' = W + BA, que al desplegar se puede fusionar con W.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P10
- P25


## 4. Intuición

Para adaptar un modelo enorme a tu tarea no hace falta reescribirlo entero: basta con una nota al margen. LoRA aprende esa nota —pequeña— y deja el original intacto.


## 5. Concepto mínimo

```text
Ajuste completo:  W' = W_entrenada           d×d parámetros por matriz
LoRA          :  W' = W + B·A               2·d·r parámetros,  r ≪ d

    W congelada · B ∈ ℝ^{d×r} · A ∈ ℝ^{r×d}
```

Al desplegar, B·A se **suma** a W: no queda coste extra en inferencia.


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('lora', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. Con d=128 y r=4, ¿cuántos parámetros se entrenan frente al ajuste completo?
2. ¿Cuántas copias del modelo hacen falta para diez tareas?
3. ¿Qué coste añade en inferencia?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('lora', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('lora', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con rango 4 se entrena una fracción diminuta de los parámetros. Y como la matriz base queda congelada, **una sola copia del modelo sirve para todas las tareas**: cada una aporta solo su adaptador. En inferencia, cero coste añadido porque BA se fusiona con W.


## 10. Comentario pedagógico

La hipótesis de fondo es empírica: que la actualización útil para adaptar un modelo es de rango bajo. No está garantizada para toda tarea, y elegir r y a qué matrices aplicarlo son decisiones que el paper estudia con ablaciones.


## 11. Error o anti-patrón deliberado

Anti-patrón: subir r «por si acaso» hasta que deja de haber ahorro.


In [ ]:
d = 4096
for r in (1, 8, 64, 512, 2048):
    lora, completo = 2 * d * r, d * d
    print(f'r={r:>4} → {lora:>10,} params ({lora/completo:>6.1%} del completo)'
          + ('  ← ya no ahorra' if lora > completo * 0.5 else ''))

## 12. Corrección

El criterio correcto es empírico y barato de obtener:


In [ ]:
protocolo = {'1': 'empezar con r pequeno (4-16)',
             '2': 'subir r solo si la metrica de validacion mejora',
             '3': 'reportar r junto con el resultado, siempre',
             '4': 'probar tambien a QUE matrices aplicarlo, no solo con que rango'}
show(protocolo)

## 13. Desafío guiado

Comprueba que una actualización de rango 2 se representa exactamente con r=2 y no con r=1.


In [ ]:
r = run_paper_lab('lora', seed=3)['result']
show(r)

## 14. Desafío autónomo

Ajusta un modelo abierto pequeño con LoRA a varios rangos sobre la misma tarea. Compara métrica, parámetros entrenados y tiempo. Localiza el rango donde deja de mejorar.


## 15. Evidencia de aprendizaje

Guarda la tabla de parámetros por rango, la forma de la actualización factorizada y tu protocolo de elección de r.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P48_lora/README.md) · evaluación formal: [`assessments/papers/P48_lora.md`](../../assessments/papers/P48_lora.md)


## 16. Cierre

Ya se adapta barato. Falta que el modelo base quepa en la máquina.


## 17. Conexión con el siguiente hito

- P49
- ecosistema de modelos abiertos ajustados

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
